In [ ]:
import pandas as pd
import numpy as np
import json
import yaml
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Paleta -----------------------------------------------------------------------
BG_DARK   = '#0f1923'
BG_CARD   = '#162030'
BG_BORDER = '#243347'
C_TEXT    = '#e8eef5'
C_MUTED   = '#5a7080'
C_GREEN   = '#27ae60'
C_YELLOW  = '#e67e22'
C_RED     = '#e74c3c'
C_BLUE    = '#3498db'
C_ORANGE  = '#FFB347'
C_PURPLE  = '#9b59b6'
C_CYAN    = '#1abc9c'

def ts_str(t):
    if pd.isna(t): return None
    return t.isoformat() if hasattr(t, 'isoformat') else str(t)

# ══════════════════════════════════════════════════════════════════════
# Configuração — altere aqui para adaptar a outra máquina
# ══════════════════════════════════════════════════════════════════════
NOME_MAQUINA    = "IC-I1-14"
CUSTO_POR_TROCA = 2614
# ══════════════════════════════════════════════════════════════════════

HERE = Path('.')

# ── Parâmetros do trigger — lidos do config.yaml (fonte de verdade) ───────────
_cfg_path = HERE / '../config.yaml'
if _cfg_path.exists():
    with open(_cfg_path) as _f:
        _cfg = yaml.safe_load(_f)
    _trig = _cfg.get('trigger', {})
else:
    _trig = {}

LIMIAR_CRITICO = _trig.get('limiar_p_risk',    0.48)
LIMIAR_AMARELO = _trig.get('amarelo_p_risk',   0.35)
FORCA_MIN_N    = _trig.get('proj_48h_limiar', 800.0)
IDADE_MIN_DIAS = _trig.get('idade_minima_dias',  15)

# ── Dados canônicos gerados por pipeline_producao.ipynb ──────────────────────
# Pré-requisito: pipeline_producao.ipynb deve ter sido executado antes.
_sinais = pd.read_csv(HERE / '02_sinais_forca.csv', parse_dates=['Timestamp'])
_vida   = pd.read_csv(HERE / '01_vida_rul.csv',     parse_dates=['Timestamp'])

_sinais['Timestamp'] = pd.to_datetime(_sinais['Timestamp'], utc=True)
_vida['Timestamp']   = pd.to_datetime(_vida['Timestamp'],   utc=True)

# Merge sinais + RUL condicional Weibull
_df = _sinais.merge(
    _vida[['Timestamp', 'rul_p10', 'rul_p50', 'rul_p90']],
    on='Timestamp', how='left',
).sort_values('Timestamp').reset_index(drop=True)

# Colunas derivadas para o dashboard
FATOR_MIN = 0.30   # p_risk=1.0 → RUL = 30% do prior Weibull
FATOR_MAX = 1.00   # p_risk=0.0 → RUL inalterado
_df['fator_deg']           = FATOR_MAX - _df['p_risk'].fillna(0) * (FATOR_MAX - FATOR_MIN)
_df['score_degradacao_v2'] = _df['p_risk']
_df['pred_rul_p10_v2']     = _df['rul_p10'] * _df['fator_deg']
_df['pred_rul_p50_v2']     = _df['rul_p50'] * _df['fator_deg']
_df['pred_rul_p90_v2']     = _df['rul_p90'] * _df['fator_deg']

def _status(p):
    if pd.isna(p):           return 'SEM_DADO'
    if p >= LIMIAR_CRITICO:  return '🔴 CRITICO'
    if p >= LIMIAR_AMARELO:  return '🟡 ATENCAO'
    return '🟢 NORMAL'

_df['pred_status_v2'] = _df['p_risk'].apply(_status)
_df['ts']             = _df['Timestamp']

df7 = _df   # nome mantido — usado pelas células de visualização

# Trocas confirmadas (SAP IW38)
trocas_raw = pd.read_csv(HERE / 'troca_modulo.csv', encoding='utf-8-sig')
trocas_all = pd.to_datetime(trocas_raw.iloc[:, 0], utc=True).sort_values().tolist()

# Parâmetros Weibull — refit automático pelo gen_vida_rul
with open(HERE / '01_weibull_params.json') as f:
    wp = json.load(f)

# ── Métricas-chave ───────────────────────────────────────────────────────────
LAST       = df7.iloc[-1]
TS_LAST    = LAST['ts']
SCORE_ATU  = LAST['score_degradacao_v2']
STATUS_ATU = LAST['pred_status_v2']
H_CICLO    = LAST['horas_desde_troca']
DIAS_CICLO = H_CICLO / 24

RUL_P10 = LAST['pred_rul_p10_v2']
RUL_P50 = LAST['pred_rul_p50_v2']
RUL_P90 = LAST['pred_rul_p90_v2']

PROJ_P10 = TS_LAST + pd.Timedelta(hours=RUL_P10)
PROJ_P50 = TS_LAST + pd.Timedelta(hours=RUL_P50)
PROJ_P90 = TS_LAST + pd.Timedelta(hours=RUL_P90)

TROCA_APR       = trocas_all[-1]
WIN_START       = PROJ_P10
WIN_END         = PROJ_P90
TROCA_APR_STR   = TROCA_APR.strftime('%d/%m/%Y')
TROCA_APR_CURTA = TROCA_APR.strftime('%d/%m')
DATA_INICIO_STR = df7['ts'].min().strftime('%b %Y')
DATA_FIM_STR    = df7['ts'].max().strftime('%b %Y')
VIDA_MED_DIAS   = wp['vida_p50_h'] / 24

# Antecedência: quando p_risk cruzou CRÍTICO antes da última troca
_win_pre     = df7[(df7['ts'] >= TROCA_APR - pd.Timedelta(days=30)) & (df7['ts'] < TROCA_APR)]
_critico_pre = _win_pre[_win_pre['score_degradacao_v2'] >= LIMIAR_CRITICO]
if len(_critico_pre) > 0:
    TS_CRITICO_PRE   = _critico_pre['ts'].iloc[0]
    HORAS_ANTES      = (TROCA_APR - TS_CRITICO_PRE).total_seconds() / 3600
    ANTECEDENCIA_STR = f"{HORAS_ANTES:.0f} horas" if HORAS_ANTES < 48 else f"{HORAS_ANTES/24:.1f} dias"
    DIA_CRITICO_STR  = TS_CRITICO_PRE.strftime('%d/%m')
else:
    TS_CRITICO_PRE   = None
    HORAS_ANTES      = 0
    ANTECEDENCIA_STR = "horas"
    DIA_CRITICO_STR  = "—"

n_trocas_total     = len(trocas_all)
ciclos             = pd.Series(trocas_all).diff().dt.total_seconds() / 3600
n_prematuras       = int((ciclos < wp['vida_p50_h'] * 0.5).sum())
custo_desperdicado = n_prematuras * CUSTO_POR_TROCA

In [2]:
header_html = f'''
<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700&display=swap');
  body, .jp-OutputArea {{ background: {BG_DARK} !important; }}
  .story-header {{
    background: linear-gradient(135deg, #0f1923 0%, #162a3f 100%);
    border: 1px solid {BG_BORDER};
    border-radius: 12px;
    padding: 32px 40px;
    margin-bottom: 8px;
    font-family: Inter, sans-serif;
  }}
  .story-header .eyebrow {{
    font-size: 11px;
    font-weight: 600;
    letter-spacing: 2px;
    color: {C_CYAN};
    text-transform: uppercase;
    margin-bottom: 8px;
  }}
  .story-header h1 {{
    font-size: 28px;
    font-weight: 700;
    color: {C_TEXT};
    margin: 0 0 8px 0;
  }}
  .story-header .subtitle {{
    font-size: 14px;
    color: {C_MUTED};
    max-width: 620px;
    line-height: 1.6;
  }}
  .story-header .data-tag {{
    display: inline-block;
    background: {BG_BORDER};
    color: {C_MUTED};
    font-size: 11px;
    padding: 3px 10px;
    border-radius: 4px;
    margin-top: 16px;
    margin-right: 6px;
  }}
</style>
<div class="story-header">
  <div class="eyebrow">Projeto Palantir · {NOME_MAQUINA} · Força de Selagem Frontal</div>
  <h1>Maintacker — Vida Útil & Impacto na Qualidade</h1>
  <div class="subtitle">
    A força de selagem frontal é influenciada por dois fatores independentes:
    o <strong style="color:{C_RED}">desgaste do maintacker</strong> (mecânico, progressivo)
    e a <strong style="color:{C_BLUE}">dosagem de adesivo</strong> (operacional, ajustável).
    Este painel conta a história do que os dados revelam — e o que eles estão prevendo agora.
  </div>
  <span class="data-tag">📅 Dados: {DATA_INICIO_STR} – {DATA_FIM_STR}</span>
  <span class="data-tag">🔩 {n_trocas_total} trocas registradas (IW38)</span>
  <span class="data-tag">⚙️ Máquina {NOME_MAQUINA}</span>
</div>
'''
display(HTML(header_html))

In [ ]:
def status_color(s):
    if 'CRITICO' in s: return C_RED
    if 'ATENCAO' in s: return C_YELLOW
    return C_GREEN

score_pct    = f"{SCORE_ATU*100:.0f}%"
status_label = STATUS_ATU.replace('🔴 ','').replace('🟡 ','').replace('🟢 ','')
s_color      = status_color(STATUS_ATU)
proj_p50_str = PROJ_P50.strftime('%d/%m/%Y')
dias_str     = f"{int(DIAS_CICLO)} dias"

kpi_html = f'''
<style>
  .kpi-grid {{
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 12px;
    margin-bottom: 8px;
    font-family: Inter, sans-serif;
  }}
  .kpi-card {{
    background: {BG_CARD};
    border: 1px solid {BG_BORDER};
    border-radius: 10px;
    padding: 20px 22px;
    position: relative;
    overflow: hidden;
  }}
  .kpi-card .accent-bar {{
    position: absolute;
    top: 0; left: 0; right: 0;
    height: 3px;
  }}
  .kpi-card .label {{
    font-size: 11px;
    font-weight: 600;
    letter-spacing: 1px;
    text-transform: uppercase;
    color: {C_MUTED};
    margin-bottom: 10px;
  }}
  .kpi-card .value {{
    font-size: 28px;
    font-weight: 700;
    line-height: 1;
    margin-bottom: 6px;
  }}
  .kpi-card .sub {{
    font-size: 12px;
    color: {C_MUTED};
    line-height: 1.4;
  }}
</style>
<div class="kpi-grid">
  <div class="kpi-card">
    <div class="accent-bar" style="background:{s_color}"></div>
    <div class="label">Status Atual</div>
    <div class="value" style="color:{s_color}">{status_label}</div>
    <div class="sub">Score: {score_pct} (limiar crítico: {LIMIAR_CRITICO*100:.0f}%)</div>
  </div>
  <div class="kpi-card">
    <div class="accent-bar" style="background:{C_BLUE}"></div>
    <div class="label">Ciclo Atual</div>
    <div class="value" style="color:{C_BLUE}">{dias_str}</div>
    <div class="sub">Desde a troca de {TROCA_APR_STR}<br>Vida mediana histórica: {VIDA_MED_DIAS:.0f} dias</div>
  </div>
  <div class="kpi-card">
    <div class="accent-bar" style="background:{C_ORANGE}"></div>
    <div class="label">Próxima Troca Estimada</div>
    <div class="value" style="color:{C_ORANGE}">{proj_p50_str}</div>
    <div class="sub">Cenário mediano (P50)<br>Janela: {WIN_START.strftime("%d/%m")} – {WIN_END.strftime("%d/%m")}</div>
  </div>
  <div class="kpi-card">
    <div class="accent-bar" style="background:{C_YELLOW}"></div>
    <div class="label">Trocas Prematuras Estimadas</div>
    <div class="value" style="color:{C_YELLOW}">{n_prematuras}</div>
    <div class="sub">Trocas antes de 50% da vida mediana<br>≈ R$ {custo_desperdicado:,.0f} em peças</div>
  </div>
</div>
'''
display(HTML(kpi_html))

In [ ]:
section_html = f'''
<div style="font-family:Inter,sans-serif; margin: 24px 0 4px 0;">
  <div style="font-size:11px;font-weight:600;letter-spacing:2px;color:{C_CYAN};text-transform:uppercase;margin-bottom:4px;">
    Seção 1 · O Passado
  </div>
  <div style="font-size:18px;font-weight:700;color:{C_TEXT};margin-bottom:4px;">
    A Impressão Digital do Desgaste — {DATA_INICIO_STR} – {DATA_FIM_STR}
  </div>
  <div style="font-size:13px;color:{C_MUTED};max-width:680px;line-height:1.6;">
    Cada linha vertical representa uma troca confirmada no SAP (IW38).
    O painel inferior mostra o <b style="color:{C_ORANGE}">score de degradação</b> do modelo —
    quando ele cruza {LIMIAR_CRITICO:.2f} (linha vermelha), o sistema entra em CRÍTICO.
    Observe como o score oscila ao longo dos ciclos e se eleva antes de cada troca genuína.
  </div>
</div>
'''
display(HTML(section_html))

ev = df7.copy()

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.55, 0.45],
    vertical_spacing=0.03,
)

fig.add_trace(go.Scatter(
    x=ev['ts'].apply(ts_str),
    y=ev['Media'],
    mode='lines',
    name='Força de Selagem (N)',
    line=dict(color=C_BLUE, width=0.8),
    opacity=0.7,
    fill='tozeroy',
    fillcolor='rgba(52,152,219,0.04)',
    hovertemplate='%{x|%d/%m %H:%M}<br>Força: <b>%{y:.0f} N</b><extra></extra>',
), row=1, col=1)

fig.add_hline(y=FORCA_MIN_N, line_dash='dot', line_color=C_RED, line_width=1,
              opacity=0.4, row=1, col=1)

score_colors = [
    C_RED    if s >= LIMIAR_CRITICO else
    C_YELLOW if s >= LIMIAR_AMARELO else
    C_GREEN
    for s in ev['score_degradacao_v2'].fillna(0)
]
fig.add_trace(go.Bar(
    x=ev['ts'].apply(ts_str),
    y=ev['score_degradacao_v2'],
    name='Score Degradação',
    marker_color=score_colors,
    opacity=0.8,
    hovertemplate='%{x|%d/%m}<br>Score: <b>%{y:.2f}</b><extra></extra>',
), row=2, col=1)

for i, dt in enumerate(trocas_all):
    is_last = (dt == trocas_all[-1])
    fig.add_vline(
        x=ts_str(dt),
        line_width=1.5 if is_last else 1,
        line_dash='solid',
        line_color=C_ORANGE if is_last else 'rgba(255,179,71,0.35)',
        opacity=1.0,
        row=1, col=1,
    )

fig.add_hline(y=LIMIAR_CRITICO, line_dash='dash', line_color=C_RED,
              line_width=1, opacity=0.5, row=2, col=1)
fig.add_hline(y=LIMIAR_AMARELO, line_dash='dot', line_color=C_YELLOW,
              line_width=1, opacity=0.4, row=2, col=1)

ax = dict(gridcolor=BG_BORDER, linecolor=BG_BORDER,
          zeroline=False, tickfont=dict(size=9, color=C_MUTED))

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor=BG_DARK,
    plot_bgcolor=BG_CARD,
    height=480,
    margin=dict(l=60, r=20, t=40, b=30),
    showlegend=False,
    hovermode='x unified',
    title=dict(
        text=f'Força de Selagem Frontal · {NOME_MAQUINA} ({DATA_INICIO_STR} – {DATA_FIM_STR})'
             f'<br><sup style="color:#3a4a60">barras laranja = trocas IW38 · última troca: {TROCA_APR_STR}</sup>',
        font=dict(size=13, color=C_TEXT),
        x=0.01, xanchor='left',
    ),
    annotations=[
        dict(x=ts_str(trocas_all[-1]), y=2200, xref='x', yref='y',
             text=TROCA_APR_CURTA, showarrow=False,
             font=dict(color=C_ORANGE, size=9, family='Inter'),
             xanchor='left', yanchor='bottom'),
        dict(x=ts_str(ev['ts'].iloc[20]), y=FORCA_MIN_N, xref='x', yref='y',
             text=f'{FORCA_MIN_N:.0f}N limiar', showarrow=False,
             font=dict(color=C_RED, size=9),
             xanchor='left', yanchor='bottom'),
        dict(x=ts_str(ev['ts'].iloc[50]), y=LIMIAR_CRITICO + 0.02, xref='x2', yref='y2',
             text=f'CRÍTICO (≥{LIMIAR_CRITICO:.2f})', showarrow=False,
             font=dict(color=C_RED, size=9),
             xanchor='left', yanchor='bottom'),
    ],
)
fig.update_xaxes(row=1, col=1, **ax)
fig.update_yaxes(row=1, col=1, title_text='Força (N)',
                 title_font=dict(size=10, color=C_MUTED), range=[0, 2500], **ax)
fig.update_xaxes(row=2, col=1, **ax)
fig.update_yaxes(row=2, col=1, title_text='Score',
                 title_font=dict(size=10, color=C_MUTED), range=[0, 1.05], **ax)
fig.show()

In [ ]:
section2_html = f'''
<div style="font-family:Inter,sans-serif; margin: 28px 0 4px 0;">
  <div style="font-size:11px;font-weight:600;letter-spacing:2px;color:{C_CYAN};text-transform:uppercase;margin-bottom:4px;">
    Seção 2 · A Confirmação
  </div>
  <div style="font-size:18px;font-weight:700;color:{C_TEXT};margin-bottom:4px;">
    "O Sinal Sabia" — Troca de {TROCA_APR_STR}
  </div>
  <div style="font-size:13px;color:{C_MUTED};max-width:680px;line-height:1.6;">
    Zoom nas 3 semanas anteriores à troca de {TROCA_APR_CURTA}.
    O modelo entrou em <b style="color:{C_RED}">CRÍTICO</b> em {DIA_CRITICO_STR} —
    <b style="color:{C_TEXT}">{ANTECEDENCIA_STR} antes</b> da troca ser registrada no SAP.
    O score de degradação sustentou-se acima de {LIMIAR_CRITICO*100:.0f}% nos dias anteriores ao evento.
    Isso confirma: <em style="color:{C_CYAN}">o dado antecede a decisão operacional</em>.
  </div>
</div>
'''
display(HTML(section2_html))

t0   = TROCA_APR - pd.Timedelta(days=21)
t1   = TROCA_APR + pd.Timedelta(days=4)
zoom = df7[(df7['ts'] >= t0) & (df7['ts'] <= t1)].copy()

fig2 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.5, 0.5],
    vertical_spacing=0.04,
)

fig2.add_trace(go.Scatter(
    x=zoom['ts'].apply(ts_str),
    y=zoom['Media'],
    mode='lines+markers',
    name='Força (N)',
    line=dict(color=C_BLUE, width=1.5),
    marker=dict(size=4, color=C_BLUE),
    hovertemplate='%{x|%d/%m %H:%M}<br>Força: <b>%{y:.0f} N</b><extra></extra>',
), row=1, col=1)

fig2.add_hline(y=FORCA_MIN_N, line_dash='dot', line_color=C_RED,
               line_width=1.2, opacity=0.5, row=1, col=1)

score_z = zoom['score_degradacao_v2'].fillna(0)
fig2.add_trace(go.Scatter(
    x=zoom['ts'].apply(ts_str),
    y=score_z * 100,
    mode='lines',
    name='Score (%)',
    line=dict(color=C_ORANGE, width=2),
    fill='tozeroy',
    fillcolor='rgba(255,179,71,0.08)',
    hovertemplate='%{x|%d/%m %H:%M}<br>Score: <b>%{y:.1f}%</b><extra></extra>',
), row=2, col=1)

fig2.add_hline(y=LIMIAR_CRITICO * 100, line_dash='dash', line_color=C_RED,
               line_width=1.5, opacity=0.7, row=2, col=1)

fig2.add_vrect(
    x0=ts_str(TROCA_APR), x1=ts_str(t1),
    fillcolor='rgba(39,174,96,0.07)',
    layer='below', line_width=0,
    row=2, col=1,
)

fig2.add_vline(x=ts_str(TROCA_APR), line_width=2.5, line_dash='solid',
               line_color=C_ORANGE, opacity=0.9, row=1, col=1)
fig2.add_vline(x=ts_str(TROCA_APR), line_width=2.5, line_dash='solid',
               line_color=C_ORANGE, opacity=0.9, row=2, col=1)

if TS_CRITICO_PRE is not None:
    fig2.add_vline(x=ts_str(TS_CRITICO_PRE), line_width=1.5, line_dash='dot',
                   line_color=C_RED, opacity=0.8, row=2, col=1)

ax2 = dict(gridcolor=BG_BORDER, linecolor=BG_BORDER,
           zeroline=False, tickfont=dict(size=10, color=C_MUTED))

fig2.update_layout(
    template='plotly_dark',
    paper_bgcolor=BG_DARK,
    plot_bgcolor=BG_CARD,
    height=440,
    margin=dict(l=65, r=20, t=50, b=30),
    showlegend=False,
    hovermode='x unified',
    title=dict(
        text=f'Zoom: Troca de {TROCA_APR_STR} — 21 dias de contexto'
             f'<br><sup style="color:#3a4a60">laranja = dia da troca · vermelho pontilhado = entrada em CRÍTICO ({DIA_CRITICO_STR})</sup>',
        font=dict(size=13, color=C_TEXT),
        x=0.01, xanchor='left',
    ),
    annotations=[
        dict(x=ts_str(TROCA_APR), y=2300, xref='x', yref='y',
             text=f'TROCA<br>{TROCA_APR_CURTA}', showarrow=False,
             font=dict(color=C_ORANGE, size=10, family='Inter'),
             xanchor='left', yanchor='bottom'),
        dict(x=ts_str(TROCA_APR), y=102, xref='x2', yref='y2',
             text='Troca', showarrow=False,
             font=dict(color=C_ORANGE, size=10),
             xanchor='left', yanchor='bottom'),
        dict(x=ts_str(TROCA_APR + pd.Timedelta(days=1)), y=50, xref='x2', yref='y2',
             text='Recovery →', showarrow=False,
             font=dict(color=C_GREEN, size=10),
             xanchor='left', yanchor='middle'),
    ],
)
fig2.update_xaxes(row=1, col=1, **ax2)
fig2.update_yaxes(row=1, col=1, title_text='Força (N)',
                  title_font=dict(size=10, color=C_MUTED), **ax2)
fig2.update_xaxes(row=2, col=1, **ax2)
fig2.update_yaxes(row=2, col=1, title_text='Score Degradação (%)',
                  title_font=dict(size=10, color=C_MUTED), range=[0, 110], **ax2)
fig2.show()

In [ ]:
section3_html = f'''
<div style="font-family:Inter,sans-serif; margin: 28px 0 4px 0;">
  <div style="font-size:11px;font-weight:600;letter-spacing:2px;color:{C_CYAN};text-transform:uppercase;margin-bottom:4px;">
    Seção 3 · Agora
  </div>
  <div style="font-size:18px;font-weight:700;color:{C_TEXT};margin-bottom:4px;">
    O Vetor Atual — O Mesmo Cenário de Antes de {TROCA_APR_CURTA}
  </div>
  <div style="font-size:13px;color:{C_MUTED};max-width:680px;line-height:1.6;">
    Desde a troca de {TROCA_APR_CURTA}, o score se recuperou mas voltou a subir — e hoje está em
    <b style="color:{C_RED}">CRÍTICO ({SCORE_ATU*100:.0f}%)</b> novamente.
    A projeção central (P50) aponta para <b style="color:{C_ORANGE}">~{PROJ_P50.strftime("%d/%m")}</b>,
    dentro da janela estimada de <b style="color:{C_CYAN}">{WIN_START.strftime("%d/%m")} – {WIN_END.strftime("%d/%m")}</b>.
    <br><br>
    <em>⚠️ Se a força de selagem não se recuperar nas próximas semanas,
    o comportamento vetorial é consistente com os ciclos que antecederam trocas genuínas passadas.</em>
  </div>
</div>
'''
display(HTML(section3_html))

t_ciclo_start = TROCA_APR - pd.Timedelta(days=2)
atual = df7[df7['ts'] >= t_ciclo_start].copy()

fig3 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.5, 0.5],
    vertical_spacing=0.04,
)

fig3.add_trace(go.Scatter(
    x=atual['ts'].apply(ts_str),
    y=atual['Media'],
    mode='lines+markers',
    name='Força (N)',
    line=dict(color=C_BLUE, width=1.5),
    marker=dict(size=4),
    hovertemplate='%{x|%d/%m %H:%M}<br>Força: <b>%{y:.0f} N</b><extra></extra>',
), row=1, col=1)

fig3.add_hline(y=FORCA_MIN_N, line_dash='dot', line_color=C_RED,
               line_width=1, opacity=0.4, row=1, col=1)

score_colors3 = [
    C_RED    if s >= LIMIAR_CRITICO else
    C_YELLOW if s >= LIMIAR_AMARELO else
    C_GREEN
    for s in atual['score_degradacao_v2'].fillna(0)
]
fig3.add_trace(go.Scatter(
    x=atual['ts'].apply(ts_str),
    y=atual['score_degradacao_v2'] * 100,
    mode='lines',
    name='Score (%)',
    line=dict(color=C_ORANGE, width=2),
    fill='tozeroy',
    fillcolor='rgba(255,179,71,0.08)',
    hovertemplate='%{x|%d/%m %H:%M}<br>Score: <b>%{y:.1f}%</b><extra></extra>',
), row=2, col=1)

fig3.add_hline(y=LIMIAR_CRITICO * 100, line_dash='dash', line_color=C_RED,
               line_width=1.5, opacity=0.7, row=2, col=1)

fig3.add_vrect(x0=ts_str(WIN_START), x1=ts_str(WIN_END),
               fillcolor='rgba(231,76,60,0.08)', layer='below', line_width=0, row=1, col=1)
fig3.add_vrect(x0=ts_str(WIN_START), x1=ts_str(WIN_END),
               fillcolor='rgba(231,76,60,0.08)', layer='below', line_width=0, row=2, col=1)

fig3.add_vline(x=ts_str(PROJ_P50), line_width=2, line_dash='dash',
               line_color=C_ORANGE, opacity=0.85, row=1, col=1)
fig3.add_vline(x=ts_str(PROJ_P50), line_width=2, line_dash='dash',
               line_color=C_ORANGE, opacity=0.85, row=2, col=1)

fig3.add_vline(x=ts_str(TROCA_APR), line_width=2, line_dash='solid',
               line_color=C_GREEN, opacity=0.6, row=1, col=1)
fig3.add_vline(x=ts_str(TROCA_APR), line_width=2, line_dash='solid',
               line_color=C_GREEN, opacity=0.6, row=2, col=1)

ax3 = dict(gridcolor=BG_BORDER, linecolor=BG_BORDER,
           zeroline=False, tickfont=dict(size=10, color=C_MUTED))

win_label = f'{WIN_START.strftime("%d/%m")} – {WIN_END.strftime("%d/%m")}'

fig3.update_layout(
    template='plotly_dark',
    paper_bgcolor=BG_DARK,
    plot_bgcolor=BG_CARD,
    height=460,
    margin=dict(l=65, r=20, t=50, b=30),
    showlegend=False,
    hovermode='x unified',
    title=dict(
        text=f'Ciclo Atual: {TROCA_APR_CURTA} → hoje + projeção de vida residual'
             f'<br><sup style="color:#3a4a60">verde = troca {TROCA_APR_CURTA} · laranja tracejado = próxima troca estimada · zona vermelha = janela {win_label}</sup>',
        font=dict(size=13, color=C_TEXT),
        x=0.01, xanchor='left',
    ),
    annotations=[
        dict(x=ts_str(TROCA_APR), y=2300, xref='x', yref='y',
             text=TROCA_APR_CURTA, showarrow=False,
             font=dict(color=C_GREEN, size=10),
             xanchor='left', yanchor='bottom'),
        dict(x=ts_str(PROJ_P50), y=2300, xref='x', yref='y',
             text='~' + PROJ_P50.strftime('%d/%m'), showarrow=False,
             font=dict(color=C_ORANGE, size=10),
             xanchor='left', yanchor='bottom'),
        dict(x=ts_str(WIN_START + (WIN_END - WIN_START) / 2), y=105, xref='x2', yref='y2',
             text=win_label, showarrow=False,
             font=dict(color=C_RED, size=10, family='Inter'),
             xanchor='center', yanchor='bottom'),
    ],
)
fig3.update_xaxes(row=1, col=1, **ax3)
fig3.update_yaxes(row=1, col=1, title_text='Força (N)',
                  title_font=dict(size=10, color=C_MUTED), **ax3)
fig3.update_xaxes(row=2, col=1, **ax3)
fig3.update_yaxes(row=2, col=1, title_text='Score (%)',
                  title_font=dict(size=10, color=C_MUTED), range=[0, 110], **ax3)
fig3.show()

In [ ]:
section4_html = f'''
<div style="font-family:Inter,sans-serif; margin: 28px 0 16px 0;">
  <div style="font-size:11px;font-weight:600;letter-spacing:2px;color:{C_CYAN};text-transform:uppercase;margin-bottom:4px;">
    Seção 4 · O Diagnóstico
  </div>
  <div style="font-size:18px;font-weight:700;color:{C_TEXT};margin-bottom:12px;">
    Maintacker ou Adesivo? — Como o Dado Distingue os Dois
  </div>

  <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;max-width:860px;">
    <div style="background:{BG_CARD};border:1px solid {C_RED}40;border-radius:10px;padding:22px;">
      <div style="font-size:13px;font-weight:700;color:{C_RED};margin-bottom:12px;">
        🔴 Assinatura do Maintacker Desgastado
      </div>
      <table style="width:100%;font-size:12px;color:{C_MUTED};border-collapse:collapse;">
        <tr style="color:{C_TEXT}"><td style="padding:5px 0;font-weight:600">Indicador</td><td style="padding:5px 0;font-weight:600">Comportamento</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Tendência (slope 14d)</td><td style="color:{C_RED}">Negativa e persistente (&gt;7 dias)</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Pior média 7d</td><td style="color:{C_RED}">Abaixo de {FORCA_MIN_N:.0f}N com frequência crescente</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Ratio (atual/médiaM14)</td><td style="color:{C_RED}">Abaixo de 1,0 e caindo</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Tempo no ciclo</td><td style="color:{C_YELLOW}">&gt; 20–30 dias (desgaste natural)</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Variabilidade A/B</td><td style="color:{C_MUTED}">Estável ou ligeiramente crescente</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Score V2</td><td style="color:{C_RED}">≥ {LIMIAR_CRITICO:.2f} por múltiplas leituras</td></tr>
      </table>
      <div style="margin-top:14px;font-size:12px;color:{C_MUTED};border-top:1px solid {BG_BORDER};padding-top:10px;">
        <b style="color:{C_TEXT}">Ação:</b> Agendar troca. Não ajustar adesivo — não resolve.
      </div>
    </div>

    <div style="background:{BG_CARD};border:1px solid {C_BLUE}40;border-radius:10px;padding:22px;">
      <div style="font-size:13px;font-weight:700;color:{C_BLUE};margin-bottom:12px;">
        🔵 Assinatura do Adesivo Desregulado
      </div>
      <table style="width:100%;font-size:12px;color:{C_MUTED};border-collapse:collapse;">
        <tr style="color:{C_TEXT}"><td style="padding:5px 0;font-weight:600">Indicador</td><td style="padding:5px 0;font-weight:600">Comportamento</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Tendência (slope 14d)</td><td style="color:{C_YELLOW}">Negativa mas de curta duração (&lt;4 dias)</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Pior média 7d</td><td style="color:{C_YELLOW}">Queda abrupta, sem gradualidade</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Ratio (atual/médiaM14)</td><td style="color:{C_YELLOW}">Queda súbita mas recuperável</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Tempo no ciclo</td><td style="color:{C_GREEN}">&lt; {IDADE_MIN_DIAS} dias (peça ainda nova)</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Variabilidade A/B</td><td style="color:{C_RED}">Alta — lados A e B divergem</td></tr>
        <tr><td style="padding:4px 0;border-top:1px solid {BG_BORDER}">Score V2</td><td style="color:{C_YELLOW}">Eleva mas oscila (sem sustentação)</td></tr>
      </table>
      <div style="margin-top:14px;font-size:12px;color:{C_MUTED};border-top:1px solid {BG_BORDER};padding-top:10px;">
        <b style="color:{C_TEXT}">Ação:</b> Verificar dosagem de adesivo. Peça ainda tem vida útil.
      </div>
    </div>
  </div>

  <div style="margin-top:16px;background:{BG_CARD};border:1px solid {BG_BORDER};border-radius:10px;padding:18px 22px;max-width:860px;">
    <div style="font-size:12px;font-weight:600;color:{C_CYAN};margin-bottom:6px;">DIAGNÓSTICO ATUAL ({TS_LAST.strftime("%d/%m/%Y")})</div>
    <div style="font-size:13px;color:{C_MUTED};line-height:1.7;">
      Ciclo em andamento: <b style="color:{C_TEXT}">{int(DIAS_CICLO)} dias</b> desde {TROCA_APR_CURTA}.
      Score: <b style="color:{C_RED}">{SCORE_ATU*100:.0f}% (CRÍTICO)</b>.
      Tempo no ciclo &gt; limiar histórico de degradação genuína.
      <br>
      <b style="color:{C_ORANGE}">Interpretação:</b> O padrão atual é consistente com desgaste mecânico.
      Ajuste de adesivo pode mascarar temporariamente o sinal, mas não altera a causa raiz.
      Próxima troca estimada: <b style="color:{C_ORANGE}">{PROJ_P50.strftime("%d/%m/%Y")}</b>
      (intervalo P10–P90: {PROJ_P10.strftime("%d/%m")} a {PROJ_P90.strftime("%d/%m")}).
    </div>
  </div>
</div>
'''
display(HTML(section4_html))

In [8]:
footer_html = f'''
<div style="font-family:Inter,sans-serif;margin-top:24px;padding:16px 20px;
            background:{BG_CARD};border:1px solid {BG_BORDER};border-radius:8px;
            display:flex;justify-content:space-between;align-items:center;">
  <div style="font-size:11px;color:{C_MUTED};">
    Dados: <code style="color:{C_CYAN}">02_sinais_forca.csv</code> · 
    <code style="color:{C_CYAN}">01_vida_rul.csv</code> · 
    <code style="color:{C_CYAN}">01_weibull_params.json</code> · 
    <code style="color:{C_CYAN}">troca_modulo.csv</code>
    &nbsp;|&nbsp; Modelo: p_risk determinístico — Weibull CDF × sinal de degradação (v2.2)
  </div>
  <div style="font-size:11px;color:{C_MUTED};">
    Projeto Palantir · {NOME_MAQUINA} · Atualizado até {TS_LAST.strftime("%d/%m/%Y")}
  </div>
</div>
'''
display(HTML(footer_html))